In [92]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
model="claude-haiku-4-5"

def add_user_message(messages, text):
    user_message = {"role":"user", "content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role":"assistant", "content":text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=0.0, stop_sequences=None):
    params = {
        "model" : model,
        "max_tokens" : 1024,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system
    message = client.messages.create(**params) 
    return message.content[0].text

In [93]:
import json

def generate_test_dataset():
    user_prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": Must include, runtime, memory size, timeout, and 
        basic structure for AWS Lambda configuration
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []

    add_user_message(messages, user_prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])

    return json.loads(text.strip())

In [94]:
dataset = generate_test_dataset()

with open("eval_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [85]:
def run_prompt(test_case):
    """Passing the input from eval dataset and running the prompt"""
    prompt = f"""
        Do the following task:
        {test_case["task"]}
        * Respond only with python, json or plain regex
        * Do not add any comments or explanation
    """
    messages=[]
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [86]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Solution criteria to consider:
<solution_criteria>
{test_case["solution_criteria"]}
</solution_criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


In [87]:
import re
import ast

def validate_json(output):
    try:
        json.loads(output.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(output):
    try:
        ast.parse(output.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(output):
    try:
        re.compile(output.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(output, test_case):
    format = test_case["format"]   
    if format=="json":
        return validate_json(output)
    elif format=="python":
        return validate_python(output)
    else:
        return validate_regex(output)


In [88]:
def run_test_case(test_case):
    """
    grading the prompt (call run_prompt()) depending on the answer generated 
    based on the input test case
    """
    output = run_prompt(test_case)

    # TO DO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score)/2

    return {
        "output":output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [89]:
from statistics import mean

def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    avg_score = mean([result["score"] for result in results])

    print(f"Average score: {avg_score}")

    return results

In [90]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.5


In [91]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport json\nimport re\n\ndef lambda_handler(event, context):\n    try:\n        message = json.loads(event['Records'][0]['Sns']['Message'])\n        topic_arn = event['Records'][0]['Sns']['TopicArn']\n        \n        pattern = r'arn:aws:sns:[^:]+:(\\d+):'\n        match = re.search(pattern, topic_arn)\n        \n        if match:\n            account_id = match.group(1)\n            return {\n                'statusCode': 200,\n                'body': json.dumps({\n                    'accountId': account_id,\n                    'topicArn': topic_arn\n                })\n            }\n        else:\n            return {\n                'statusCode': 400,\n                'body': json.dumps({\n                    'error': 'Account ID not found in ARN'\n                })\n            }\n    except Exception as e:\n        return {\n            'statusCode': 500,\n            'body': json.dumps({\n                'error': str(e)\n            })\n        }\n",